# HIL Data Integration for Fault Testing

**Goal:** Test SSL model on real HIL fault data from supervisor

**Training Strategy:**
- 3 sensors from A2D2: accelerator_pedal, vehicle_speed, brake_pressure
- 2 sensors from HIL healthy: omega_In_Trm, n_Engine (RPM)
- Total: 5 sensors

**Testing Strategy:**
- 6 HIL fault files (acc gain/noise/stuck, rpm gain/noise/stuck)
- Random selection: each run picks random fault file + random faulty sensor
- Same 5 sensors as training

In [ ]:
import numpy as np
import json
import torch
import random
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score

print("Libraries loaded successfully!")

## Part 1: Load HIL CSV Files

In [ ]:
def load_hil_csv(filepath):
    """
    Load HIL CSV file and extract sensor data.
    
    Returns:
        dict with keys: 'time', 'omega', 'rpm', 'acc_pedal', 'brake_pedal', 'velocity'
    """
    print(f"Loading HIL data from: {filepath}")
    
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    # Find where data starts (after 'trace_values' line)
    data_start_idx = None
    for i, line in enumerate(lines):
        if line.startswith('trace_values'):
            data_start_idx = i + 1
            break
    
    if data_start_idx is None:
        raise ValueError(f"Could not find 'trace_values' in {filepath}")
    
    # Parse data rows
    data = []
    for line in lines[data_start_idx:]:
        if line.strip() and line.startswith(','):
            parts = line.strip().split(',')
            if len(parts) >= 6:
                try:
                    # Columns: time, omega, rpm, acc_pedal, brake_pedal, velocity
                    row = [float(parts[i]) if parts[i] else 0.0 for i in range(1, 7)]
                    data.append(row)
                except ValueError:
                    continue
    
    data = np.array(data)
    
    result = {
        'time': data[:, 0],
        'omega': data[:, 1],
        'rpm': data[:, 2],
        'acc_pedal': data[:, 3],
        'brake_pedal': data[:, 4],
        'velocity': data[:, 5]
    }
    
    print(f"  ✅ Loaded {len(data)} samples, duration: {data[-1, 0]:.1f}s")
    return result

# Test loading
hil_healthy = load_hil_csv('healthy.csv')
print(f"\nHealthy data shape: {hil_healthy['omega'].shape}")
print(f"Sensors: omega, rpm, acc_pedal, brake_pedal, velocity")

## Part 2: Load A2D2 Data (3 sensors)

In [ ]:
# Path detection for A2D2 data
PROJECT_ROOT = Path.cwd()

# Find training data
possible_data_dirs = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / ".." / "Project" / "data",
    PROJECT_ROOT,
]

TRAINING_JSON = None
for data_dir in possible_data_dirs:
    candidate = data_dir / "Training_Dataset-20190401121727_bus_signals.json"
    if candidate.exists():
        TRAINING_JSON = candidate
        break

# Check if we need to extract ZIP
if TRAINING_JSON is None:
    TRAINING_ZIP = PROJECT_ROOT / "Training_Dataset-20190401121727_bus_signals.zip"
    if TRAINING_ZIP.exists():
        print(f"Extracting {TRAINING_ZIP.name}...")
        import zipfile
        with zipfile.ZipFile(TRAINING_ZIP, 'r') as zip_ref:
            zip_ref.extractall(PROJECT_ROOT)
        TRAINING_JSON = PROJECT_ROOT / "Training_Dataset-20190401121727_bus_signals.json"

if not TRAINING_JSON or not TRAINING_JSON.exists():
    raise FileNotFoundError("Training JSON not found!")

print(f"Found A2D2 training data: {TRAINING_JSON}")

In [ ]:
def load_a2d2_sensors(json_path, sensors=['accelerator_pedal', 'vehicle_speed', 'brake_pressure']):
    """
    Load specific sensors from A2D2 JSON file.
    
    Returns:
        dict with sensor names as keys, each containing resampled time-series
    """
    print(f"Loading A2D2 data from: {json_path}")
    
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    result = {}
    for sensor in sensors:
        if sensor not in data:
            raise ValueError(f"Sensor '{sensor}' not found in A2D2 data!")
        
        # Extract values (timestamp, value)
        values = np.array(data[sensor]['values'])
        timestamps = values[:, 0] / 1e6  # Convert microseconds to seconds
        sensor_values = values[:, 1]
        
        # Resample to 100 Hz (to match HIL)
        target_hz = 100
        duration = timestamps[-1] - timestamps[0]
        num_samples = int(duration * target_hz)
        target_times = np.linspace(timestamps[0], timestamps[-1], num_samples)
        resampled = np.interp(target_times, timestamps, sensor_values)
        
        result[sensor] = resampled
        print(f"  ✅ {sensor}: {len(resampled)} samples @ {target_hz} Hz")
    
    return result

# Load 3 sensors from A2D2
a2d2_data = load_a2d2_sensors(TRAINING_JSON)
print(f"\nA2D2 sensors loaded: {list(a2d2_data.keys())}")

## Part 3: Normalize and Combine Data (5 sensors)

In [ ]:
def normalize_brake_pressure_to_percentage(brake_pressure):
    """
    Convert brake pressure (Pa) to approximate percentage [0-100].
    Typical automotive brake pressure: 0-10 MPa (0-10,000,000 Pa)
    """
    # Normalize to 0-100 range
    max_pressure = 10_000_000  # 10 MPa typical max
    brake_percentage = (brake_pressure / max_pressure) * 100
    brake_percentage = np.clip(brake_percentage, 0, 100)
    return brake_percentage

# Normalize brake pressure
a2d2_data['brake_pressure'] = normalize_brake_pressure_to_percentage(a2d2_data['brake_pressure'])
print(f"Brake pressure converted to percentage: mean={a2d2_data['brake_pressure'].mean():.2f}%, std={a2d2_data['brake_pressure'].std():.2f}%")

In [ ]:
def combine_training_data(a2d2_data, hil_healthy, target_length=None):
    """
    Combine A2D2 (3 sensors) + HIL healthy (2 sensors) into 5-sensor dataset.
    
    Sensor order:
    0. accelerator_pedal (A2D2)
    1. vehicle_speed (A2D2)
    2. brake_pedal (A2D2, converted from pressure)
    3. omega (HIL)
    4. rpm (HIL)
    """
    # Find minimum length to align datasets
    lengths = [
        len(a2d2_data['accelerator_pedal']),
        len(a2d2_data['vehicle_speed']),
        len(a2d2_data['brake_pressure']),
        len(hil_healthy['omega']),
        len(hil_healthy['rpm'])
    ]
    min_length = min(lengths)
    
    if target_length is None:
        target_length = min_length
    
    print(f"Combining data to length: {target_length}")
    
    # Stack sensors (samples × 5 sensors)
    combined = np.column_stack([
        a2d2_data['accelerator_pedal'][:target_length],
        a2d2_data['vehicle_speed'][:target_length],
        a2d2_data['brake_pressure'][:target_length],
        hil_healthy['omega'][:target_length],
        hil_healthy['rpm'][:target_length]
    ])
    
    print(f"Combined training data shape: {combined.shape} (samples × sensors)")
    print(f"\nSensor statistics:")
    sensor_names = ['accelerator', 'speed', 'brake', 'omega', 'rpm']
    for i, name in enumerate(sensor_names):
        print(f"  {name:12s}: mean={combined[:, i].mean():10.2f}, std={combined[:, i].std():10.2f}")
    
    return combined, sensor_names

# Combine data
training_data, sensor_names = combine_training_data(a2d2_data, hil_healthy)
print(f"\n✅ Training data ready: {training_data.shape}")

## Part 4: Window and Normalize Training Data

In [ ]:
def create_windows(data, window_size=50, stride=25):
    """
    Create sliding windows from time-series data.
    
    Args:
        data: (samples, sensors)
        window_size: number of timesteps per window
        stride: step size between windows
    
    Returns:
        windows: (num_windows, window_size, sensors)
    """
    num_samples, num_sensors = data.shape
    windows = []
    
    for start in range(0, num_samples - window_size + 1, stride):
        window = data[start:start + window_size, :]
        windows.append(window)
    
    windows = np.array(windows)
    print(f"Created {len(windows)} windows of size ({window_size}, {num_sensors})")
    return windows

# Z-score normalization per sensor
mu = training_data.mean(axis=0)
sigma = training_data.std(axis=0)
training_data_normalized = (training_data - mu) / (sigma + 1e-8)

print("Normalization stats:")
for i, name in enumerate(sensor_names):
    print(f"  {name:12s}: mu={mu[i]:10.2f}, sigma={sigma[i]:10.2f}")

# Create windows
WINDOW_SIZE = 50  # 0.5 seconds at 100 Hz
STRIDE = 25       # 50% overlap

X_train = create_windows(training_data_normalized, window_size=WINDOW_SIZE, stride=STRIDE)
print(f"\nTraining windows shape: {X_train.shape} (num_windows, time, sensors)")

## Part 5: Load Pre-trained SSL Model

**Note:** You need to run the main SSL notebook first to train the model and save it.

In [ ]:
# Define the same encoder architecture from main notebook
import torch.nn as nn

class Encoder1DCNN(nn.Module):
    def __init__(self, num_sensors=5, embedding_dim=256):
        super(Encoder1DCNN, self).__init__()
        self.conv1 = nn.Conv1d(num_sensors, 64, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, embedding_dim, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(embedding_dim)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # x: (batch, time, sensors) → (batch, sensors, time)
        x = x.transpose(1, 2)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x).squeeze(-1)
        return x

# Option 1: Train new model on 5 sensors
print("\n" + "="*80)
print("OPTION 1: Train new SSL model on combined 5-sensor data")
print("="*80)
print("You need to run the SSL training from main notebook with this data.")
print("For now, we'll create a fresh encoder for testing purposes.")

# Initialize encoder
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = Encoder1DCNN(num_sensors=5, embedding_dim=256).to(DEVICE)

print(f"\nEncoder initialized on {DEVICE}")
print(f"Number of parameters: {sum(p.numel() for p in encoder.parameters()):,}")

## Part 6: Load All HIL Fault Files

In [ ]:
# Load all fault files
fault_files = {
    'acc_gain': 'acc fault gain.csv',
    'acc_noise': 'acc fault noise.csv',
    'acc_stuck': 'acc fault stuck.csv',
    'rpm_gain': 'rpm fault gain.csv',
    'rpm_noise': 'rpm fault noise.csv',
    'rpm_stuck': 'rpm fault stuck at.csv'
}

hil_faults = {}
print("Loading HIL fault files...")
for name, filepath in fault_files.items():
    hil_faults[name] = load_hil_csv(filepath)

print(f"\n✅ Loaded {len(hil_faults)} fault scenarios")

## Part 7: Random HIL Fault Injection for Testing

In [ ]:
def prepare_hil_fault_data(hil_fault_dict, fault_name, mu, sigma, window_size=50, stride=50):
    """
    Prepare HIL fault data for testing.
    
    Args:
        hil_fault_dict: HIL data dict from load_hil_csv
        fault_name: which fault scenario (for tracking ground truth)
        mu, sigma: normalization parameters from training
    
    Returns:
        windows: (num_windows, window_size, 5)
        faulty_sensor_idx: which sensor has the fault
        fault_type: type of fault
    """
    # Determine which sensor is faulty
    if 'acc' in fault_name:
        faulty_sensor_idx = 0  # accelerator is index 0
        fault_type = fault_name.split('_')[1]  # gain, noise, stuck
    elif 'rpm' in fault_name:
        faulty_sensor_idx = 4  # rpm is index 4
        fault_type = fault_name.split('_')[1]
    else:
        faulty_sensor_idx = -1
        fault_type = 'unknown'
    
    # Stack 5 sensors in same order as training
    # Order: accelerator, speed, brake, omega, rpm
    data = np.column_stack([
        hil_fault_dict['acc_pedal'],
        hil_fault_dict['velocity'],
        hil_fault_dict['brake_pedal'],
        hil_fault_dict['omega'],
        hil_fault_dict['rpm']
    ])
    
    # Normalize using training stats
    data_normalized = (data - mu) / (sigma + 1e-8)
    
    # Create windows
    windows = create_windows(data_normalized, window_size=window_size, stride=stride)
    
    return windows, faulty_sensor_idx, fault_type

# Test on one fault file
test_fault_name = 'acc_gain'
test_windows, faulty_idx, fault_type = prepare_hil_fault_data(
    hil_faults[test_fault_name], 
    test_fault_name, 
    mu, 
    sigma
)

print(f"\nTest fault: {test_fault_name}")
print(f"  Faulty sensor: {sensor_names[faulty_idx]} (index {faulty_idx})")
print(f"  Fault type: {fault_type}")
print(f"  Windows shape: {test_windows.shape}")

## Part 8: Random Fault Selection Loop

In [ ]:
def run_random_fault_test(encoder, hil_faults, mu, sigma, num_runs=15, seed=42):
    """
    Run multiple test iterations with random fault selection.
    
    Each run:
    - Randomly selects a fault file
    - Tests detection and localization
    
    Returns:
        results: list of dicts with test results
    """
    random.seed(seed)
    np.random.seed(seed)
    
    fault_names = list(hil_faults.keys())
    results = []
    
    print("\n" + "="*80)
    print(f"RANDOM FAULT TESTING: {num_runs} runs")
    print("="*80)
    
    for run_idx in range(num_runs):
        # Randomly select fault
        fault_name = random.choice(fault_names)
        
        # Prepare fault data
        fault_windows, faulty_sensor_idx, fault_type = prepare_hil_fault_data(
            hil_faults[fault_name],
            fault_name,
            mu,
            sigma,
            window_size=50,
            stride=50
        )
        
        # For now, just record what would be tested
        # (Detection/localization code will be added after SSL training)
        result = {
            'run': run_idx + 1,
            'fault_name': fault_name,
            'fault_type': fault_type,
            'faulty_sensor': sensor_names[faulty_sensor_idx],
            'faulty_sensor_idx': faulty_sensor_idx,
            'num_windows': len(fault_windows)
        }
        
        results.append(result)
        
        print(f"Run {run_idx+1:2d}: {fault_name:15s} | Faulty: {sensor_names[faulty_sensor_idx]:12s} | Windows: {len(fault_windows):4d}")
    
    return results

# Run test
test_results = run_random_fault_test(encoder, hil_faults, mu, sigma, num_runs=15)

print(f"\n✅ Random fault testing setup complete!")
print(f"\nFault type distribution:")
fault_type_counts = {}
for r in test_results:
    ft = r['fault_type']
    fault_type_counts[ft] = fault_type_counts.get(ft, 0) + 1

for ft, count in sorted(fault_type_counts.items()):
    print(f"  {ft:10s}: {count:2d} runs")

print(f"\nSensor fault distribution:")
sensor_counts = {}
for r in test_results:
    sensor = r['faulty_sensor']
    sensor_counts[sensor] = sensor_counts.get(sensor, 0) + 1

for sensor, count in sorted(sensor_counts.items()):
    print(f"  {sensor:12s}: {count:2d} runs")

## Part 9: Next Steps

**To complete the HIL integration:**

1. **Train SSL encoder** on the combined 5-sensor training data (X_train)
   - Use transformation classification pretext task
   - Train ensemble of 5 models

2. **Compute Mahalanobis statistics** on normal HIL healthy data
   - Extract embeddings from healthy windows
   - Compute mean and covariance

3. **Test on HIL faults**
   - Run detection (Mahalanobis distance)
   - Run localization (ablation-based)
   - Report metrics per fault type

4. **Compare results**
   - Synthetic A2D2 faults vs Real HIL faults
   - Add section to thesis: "Validation on Real HIL Data"

---

**Current Status:**
- ✅ HIL data loading
- ✅ A2D2 + HIL data combination (5 sensors)
- ✅ Brake unit conversion
- ✅ Random fault selection
- ⏳ SSL training needed
- ⏳ Detection/localization testing needed